# Pandas series intro

## Data structures

- categorical data exclude NaN 
- date time displayed and storage format can be different
- 2 ways to set datetime and categorical types
    - directly through `dtype=datetime64[ns]`
    -  `.to_datetime()`, or `astype('category')`

In [ ]:
from IPython.display import display
import pandas as pd
import numpy as np

# what is the number of category
states = ['CA', np.nan, 'NY', 'TX']
s=pd.Series(states).astype('category')
display(s.cat.categories)
len(s.cat.categories)

Index(['CA', 'NY', 'TX'], dtype='object')

3

In [2]:
# how to convert strings to datetime
string_dates_missing = ['2020-01-01 4:30', None, '2020-01-03']

pd.to_datetime(pd.Series([i[0:10] for i in string_dates_missing if i is not None]))

0   2020-01-01
1   2020-01-03
dtype: datetime64[ns]

## Intro to series

- series have indices
- filter series based on values or indices
- manipulation of series aligning indices
- categorical types make filtering easier
    - `s_cat=pd.CategoricalDtype(categories=[..], ordered=True)` (ascending order) and assign categories `s.astype(s_cat)`

| Function | Description |
|---|---|
| pd.Series(data=None, index=None, dtype=None, name=None, copy=False) | Create a Series from data (sequence, dict, or scalar). |
| s.index | Access index of the Series. |
| s.astype(dtype) | Cast Series to dtype. Use errors='ignore' to return the original on failure. |
| s[boolean_array] | Select values where boolean_array is True. |
| s.cat.categories | Return categories of a categorical Series. |

In [3]:
t_shirt=pd.Series([2, np.nan, 10, 5, 2], index=['brown', 'brown', 'blue', 'yellow', 'blue'])
display(t_shirt.index)
display(t_shirt.astype('category').cat.categories)

Index(['brown', 'brown', 'blue', 'yellow', 'blue'], dtype='object')

Index([2.0, 5.0, 10.0], dtype='float64')

In [4]:
# How many sales of blue and brown shirts?
display(t_shirt[['blue','brown']].sum())

# How many unique sale amounts?
display(t_shirt.nunique(dropna=True))

# How many sale amounts greater than 2?
display(t_shirt[t_shirt>2].sum())
 
# production costs from H to L are yellow, brown, and blue.
# How many shirts are less expensive to produce than yellow?
color_cat=pd.CategoricalDtype(categories=['yellow','brown', 'blue'], ordered=True)
expense=pd.Series(['brown', 'brown', 'blue', 'yellow', 'blue'], dtype='category')
display(sum(expense.astype(color_cat)>'brown'))



np.float64(14.0)

3

np.float64(15.0)

2

## intermediate series

### dunker methods

- index of the two series is aligned before performing math operations.
- `add()` vs `+`
    - `add(fill_value=0)` but `+` not allowed

| Method / Function | Operator | Description |
|---|---:|---|
| s.add(s2) | s + s2 | Adds series |
| s.sub(s2) | s - s2 | Subtracts series |
| s.mul(s2) / s.multiply(s2) | s * s2 | Multiplies series |
| s.div(s2) / s.truediv(s2) | s / s2 | Divides series |


In [5]:
s1 = pd.Series([10, 20, 30], index=['cn','us','us'])
s2 = pd.Series([    35, 44, 53], index=['us','us','jp'], name='s2')

# what is the total sales in the world?
# Note: s1+s2 returns NaN for cn and jp wrong results
s1+s2 

cn     NaN
jp     NaN
us    55.0
us    64.0
us    65.0
us    74.0
dtype: float64

In [6]:
s1.add(s2,fill_value=0)

cn    10.0
jp    53.0
us    55.0
us    64.0
us    65.0
us    74.0
dtype: float64

### aggregation functions for series

- directly apply for series
- used as keywords in `agg(['agg func word'])`

| Method / Property | Description |
|---|---|
| s.autocorr(lag=1) | Pearson correlation between s and shifted s (lag). |
| s.corr(other, method='pearson') | Correlation with other series ('pearson','spearman','kendall', or callable). |
| s.cov(other, min_periods=None) | Covariance with other series. |
| s.max(axis=None, skipna=None, level=None, numeric_only=None) | Maximum value. (min, sum, mean, median) |
| s.quantile(q=0.5, interpolation='linear') | Quantile(s); returns Series if q is list. |
| s.sem(axis=None, skipna=None, level=None, ddof=1, numeric_only=None) | Unbiased standard error of the mean. |
| s.var(axis=None, skipna=None, level=None, ddof=1, numeric_only=None) | Unbiased variance. (also std) |
| s.nunique(dropna=True) | Count of unique items. |
| s.count(level=None) | Count of non-missing items. |
| s.size | Number of items in series (property). |

In [7]:
# what is the mean, min and std. dev and rename as average and minimum?
# results is a series with label index
out = s1.agg(['mean', 'std', 'min'])[['mean', 'min']]
out = out.rename(index={'mean': 'average', 'min': 'minimum'})
display(out)

average    20.0
minimum    10.0
dtype: float64

In [8]:
# directly apply agg. func
pd.Series([s1.mean(), s1.std()],index=['average','stadnard deviaton'])

average              20.0
stadnard deviaton    10.0
dtype: float64

### converting data type

2 commonly used:
- `.to_datetime()`
- `.astype('string')` ('float','category')
    - `df_cat=pd.CategoricalDtype(categories=df0, sorted=True)`
    - `df['cntry']=df['cntry].astype(df_cat); df['cntry].cat.ordered`


### missing value & interpolation
- `s.isna()`
- `s.fillna(0)`
- `s.dropna()`
- `s.interpolate(method='linear')`

### dedup
- `.drop_duplicates(keep='first)`
- `s[s.duplicated(keep=False)]`
- `.value_counts(dropna=False)` count all values appearing more than once


### sort idx or val
- `s.sort_index(axis=0, ascending=True)`
- `s.sort_values(axis=0, ascending=True)`

In [ ]:
# duplicates (10,12), outlier (1000), missing (np.nan)
vals = pd.Series([10, 12, np.nan, 11, 10, 1000, 12, 13, 10, np.nan])
upper_bound=vals.quantile(0.8)
vals[vals>upper_bound]=999
display(vals)

0     10.0
1     12.0
2      NaN
3     11.0
4     10.0
5    999.0
6     12.0
7    999.0
8     10.0
9      NaN
dtype: float64

In [27]:
display(vals.drop_duplicates(keep='first'))

0     10.0
1     12.0
2      NaN
3     11.0
5    999.0
dtype: float64

In [29]:
# determine which one has duplcates
vals[vals.duplicated(keep=False)].unique()

array([ 10.,  12.,  nan, 999.])

In [35]:
# returns a series
scount=vals.value_counts(dropna=False)
display(scount[lambda i: i>1])

10.0     3
12.0     2
NaN      2
999.0    2
Name: count, dtype: int64

In [36]:
# alternatively
scount[scount>1]

10.0     3
12.0     2
NaN      2
999.0    2
Name: count, dtype: int64

### ranking

- `s.rank(axis=0, method='dense')`
    - method handles ties: 'average','min','max','first','dense'

In [5]:
from IPython.display import display
import pandas as pd
import numpy as np

df0 = pd.DataFrame({
    "firm":    ["A", "A", "A", "A", "B", "B", "B", "B", "C", "C", "C", "C"],
    "year":    [2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023, 2020, 2021, 2022, 2023],
    "region":  ["East", "East", "East", "East", "West", "West", "West", "West", "East", "East", "East", "East"],
    "sales":   [120, 135, 135, 160, 90, 110, 105, 140, 120, np.nan, 130, 130],
    "profit":  [15, 18, 18, 25, 10, 12, 11, 20, 15, 16, 19, 19]
})

In [10]:
# Within each year, rank firms by sales 
# in descending order using method="dense", 
# but only for firms in the "East" region. 
# Then create a new column that shows each East firm's rank change compared 
# with its previous year. If a prior year rank does not exist, leave it as missing.
df=df0.copy()

df.loc[df['region']=='East','sales_rank']=df.loc[df['region']=='East'].groupby('year')['sales'].rank(method='dense',ascending=False)

df


,firm,year,region,sales,profit,sales_rank
0,A,2020,East,120.0,15,1.0
1,A,2021,East,135.0,18,1.0
2,A,2022,East,135.0,18,1.0
3,A,2023,East,160.0,25,1.0
4,B,2020,West,90.0,10,NaN
5,B,2021,West,110.0,12,NaN
6,B,2022,West,105.0,11,NaN
7,B,2023,West,140.0,20,NaN
8,C,2020,East,120.0,15,1.0
9,C,2021,East,NaN,16,NaN


In [ ]:
# For each firm, rank profit across all years in ascending order using method="first". 
# Then, for rows where sales is not missing, create a flag indicating whether that row is simultaneously:
#  - the firm's best profit rank so far up to that year, and
#  - in the top 2 sales values for that firm across all years 
#       using method="min" with descending order.